In [1]:
!nvidia-smi
!nvcc --version

Sun May  3 07:06:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [19]:
!apt-get install -y libtiff-dev libomp-dev 2>&1 | tail -5
!dpkg -l libtiff-dev | grep libtiff

Building dependency tree...
Reading state information...
libomp-dev is already the newest version (1:14.0-55~exp2).
libtiff-dev is already the newest version (4.3.0-6ubuntu0.13).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
ii  libtiff-dev:amd64 4.3.0-6ubuntu0.13 amd64        Tag Image File Format library (TIFF), development files


In [4]:
#upload files

from google.colab import files
import os

os.makedirs('/content/src', exist_ok=True)
uploaded = files.upload()

for fname, data in uploaded.items():
    dest = f'/content/src/{fname}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  Saved {fname} ({len(data)//1024} KB)')

print('\nAll files in /content/src:')
!ls -lh /content/src/

Saving AsyncGPUWorker.cpp to AsyncGPUWorker.cpp
Saving AsyncGPUWorker.h to AsyncGPUWorker.h
Saving BoundedTileQueue.h to BoundedTileQueue.h
Saving CPUWorker.h to CPUWorker.h
Saving CUDAKernels.cu to CUDAKernels.cu
Saving CUDAKernels.cuh to CUDAKernels.cuh
Saving CUDAKernels.h to CUDAKernels.h
Saving CUDAKernelsStub.cpp to CUDAKernelsStub.cpp
Saving CUDAStreamManager.cpp to CUDAStreamManager.cpp
Saving CUDAStreamManager.h to CUDAStreamManager.h
Saving GPUWorker.h to GPUWorker.h
Saving IWorker.h to IWorker.h
Saving main.cpp to main.cpp
Saving OptimizedCPUWorker.cpp to OptimizedCPUWorker.cpp
Saving OptimizedCPUWorker.h to OptimizedCPUWorker.h
Saving OutputWriter.cpp to OutputWriter.cpp
Saving OutputWriter.h to OutputWriter.h
Saving Scheduler.h to Scheduler.h
Saving tiffio.h to tiffio.h
Saving tiffio_mock.cpp to tiffio_mock.cpp
Saving Tile.h to Tile.h
Saving TileOptimizer.cpp to TileOptimizer.cpp
Saving TileOptimizer.h to TileOptimizer.h
Saving TileReader.cpp to TileReader.cpp
Saving TileR

In [17]:
!find /usr -name "libtiff*" 2>/dev/null

/usr/share/doc/libtiff5
/usr/share/doc/libtiff-dev
/usr/share/doc/libtiffxx5
/usr/share/man/man3/libtiff.3tiff.gz
/usr/lib/x86_64-linux-gnu/libtiffxx.so.5.7.0
/usr/lib/x86_64-linux-gnu/libtiff.so
/usr/lib/x86_64-linux-gnu/libtiffxx.a
/usr/lib/x86_64-linux-gnu/libtiff.so.5.7.0
/usr/lib/x86_64-linux-gnu/libtiff.so.5
/usr/lib/x86_64-linux-gnu/pkgconfig/libtiff-4.pc
/usr/lib/x86_64-linux-gnu/libtiffxx.so.5
/usr/lib/x86_64-linux-gnu/libtiffxx.so
/usr/lib/x86_64-linux-gnu/libtiff.a
/usr/lib/R/site-library/pak/library/pkgdepends/sysreqs/rules/libtiff.json
/usr/local/lib/python3.12/dist-packages/nvidia/nvimgcodec/extensions/libtiff_ext.so.0.7.0
/usr/local/lib/python3.12/dist-packages/pyproj.libs/libtiff-7030d347.so.6.0.2
/usr/local/lib/python3.12/dist-packages/pillow.libs/libtiff-13a02c81.so.6.1.0
/usr/local/lib/python3.12/dist-packages/fiona.libs/libtiff-fiona-c967de8d.so.5.7.0
/usr/local/lib/python3.12/dist-packages/pygame.libs/libtiff-23a934bd.so.5.8.0
/usr/local/lib/python3.12/dist-package

In [21]:
COMPILE_CMD = """
cd /content/src && nvcc \\
  -std=c++17 \\
  -O2 \\
  -DUSE_CUDA \\
  -arch=sm_75 \\
  -Xcompiler -fopenmp \\
  -I. \\
  main.cpp \\
  TileReader.cpp \\
  transform.cpp \\
  OptimizedCPUWorker.cpp \\
  AsyncGPUWorker.cpp \\
  CUDAStreamManager.cpp \\
  VramManager.cpp \\
  TileOptimizer.cpp \\
  OutputWriter.cpp \\
  CUDAKernels.cu \\
  -L/usr/lib/x86_64-linux-gnu -ltiff \\
  -lpthread \\
  -lgomp \\
  -o /content/gigapixel_pipeline \\
  2>&1
"""
!{COMPILE_CMD}
!ls -lh /content/gigapixel_pipeline 2>/dev/null && echo 'Build OK' || echo 'Build FAILED — see errors above'

nvlink warning : Skipping incompatible '/usr/lib/x86_64-linux-gnu/libpthread.a' when searching for -lpthread
-rwxr-xr-x 1 root root 1.1M May  3 08:01 /content/gigapixel_pipeline
Build OK


In [24]:
import time, subprocess

os.chdir('/content')

print('=== GPU pipeline (grayscale) ===')
t0 = time.perf_counter()
# '1\n' = grayscale choice; '2\n' = rotate 90 CW
result = subprocess.run(
    ['/content/gigapixel_pipeline'],
    input='1\n',
    text=True,
    capture_output=True
)
elapsed = time.perf_counter() - t0

print(result.stdout[-3000:])   # last 3k chars of stdout
if result.stderr:
    print('STDERR:', result.stderr[-1000:])

W_IMG, H_IMG = 4096, 4096
megapixels = W_IMG * H_IMG / 1e6
throughput  = megapixels / elapsed
print(f'\nElapsed: {elapsed:.2f}s  |  Throughput: {throughput:.1f} MP/s')

=== GPU pipeline (grayscale) ===
]
[Main/Reader] Tile 7/12 | Read (1024,512) -> Write (1024,512)
[Worker 0] Popped tile (512,512). Requesting VRAM...
[AsyncGPUWorker] Processing tile (512,512)  512x512  size=1024 KB
[AsyncGPUWorker] GPU path unavailable, falling back to CPU
[Worker 0] Finished tile (512,512). Submitting to Writer...
[Main/Reader] Tile 8/12 | Read (1536,512) -> Write (1536,512)
[Worker 0] Popped tile (1024,512). Requesting VRAM...
[AsyncGPUWorker] Processing tile (1024,512)  512x512  size=1024 KB
[AsyncGPUWorker] GPU path unavailable, falling back to CPU
[Worker 0] Finished tile (1024,512). Submitting to Writer...
[Main/Reader] Tile 9/12 | Read (0,1024) -> Write (0,1024)
[Worker 0] Popped tile (1536,512). Requesting VRAM...
[AsyncGPUWorker] Processing tile (1536,512)  414x512  size=828 KB
[AsyncGPUWorker] GPU path unavailable, falling back to CPU
[Worker 0] Finished tile (1536,512). Submitting to Writer...
[Main/Reader] Tile 10/12 | Read (512,1024) -> Write (512,1024)
[

In [23]:
# --- Rotate test ---
print('=== GPU pipeline (rotate 90 CW) ===')
t0 = time.perf_counter()
result_r = subprocess.run(
    ['/content/gigapixel_pipeline'],
    input='2\n',
    text=True,
    capture_output=True
)
elapsed_r = time.perf_counter() - t0
print(result_r.stdout[-2000:])
print(f'Elapsed: {elapsed_r:.2f}s  |  Throughput: {megapixels/elapsed_r:.1f} MP/s')

=== GPU pipeline (rotate 90 CW) ===
ptimized rotation
[OptimizedCPUWorker] Cache hit rate: 90%
[Worker 0] Finished tile (1024,1024). Submitting to Writer...
[Main/Reader] Tile 11/12 | Read (1536,277) -> Write (512,1536)
[Worker 0] Popped tile (0,1536). Requesting VRAM...
[OptimizedCPUWorker] Processing tile (0,1536) size=414x512
[OptimizedCPUWorker] Running cache-optimized rotation
[OptimizedCPUWorker] Cache hit rate: 90%
[Worker 0] Finished tile (0,1536). Submitting to Writer...
[Worker 0] Popped tile (512,1536). Requesting VRAM...
[OptimizedCPUWorker] Processing tile (512,1536) size=414x512
[OptimizedCPUWorker] Running cache-optimized rotation
[Writer] Wrote tile at (512,0)  [2 / 0]
[Main/Reader] Tile 12/12 | Read (1536,0) -> Write (1024,1536)
[Main] All tiles read from disk. Sending 'Finished' signal to Input Queue.
[Main] Waiting for Worker threads to clear the remaining queue...
[OptimizedCPUWorker] Cache hit rate: 90%
[Worker 0] Finished tile (512,1536). Submitting to Writer...
[